# Movie Lens Recommendation

## Context
MovieLens is one of the most iconic datasets in data science, widely used in academic research and benchmarks for collaborative and content-based filtering algorithms. In a market scenario where user retention is vital, such as with Netflix or Prime Video, the ability to suggest the right content at the right time directly impacts a platform's revenue and conversion rates. This project develops a recommendation engine capable of analyzing millions of historical ratings to predict user preferences.

**Objective:** The goal of this project is to develop a recommendation system using the MovieLens dataset, simulating the logic used by companies like Netflix to maximize user engagement. The objective is to create a model capable of suggesting 5 personalized movies for a user by analyzing their past consumption behavior and identifying patterns through collaborative filtering with the K-Nearest Neighbors (KNN) algorithm.


[Click here to access the dataset ](https://grouplens.org/datasets/movielens/32m/)

### • Importing and loading data

In [1]:
# Importing libraries
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

In [2]:
# List the dataset inputs that were uploaded.
os.listdir("/kaggle/input/datasets/barlettab/movie-lens-32m")

['checksums.txt',
 'movies.csv',
 'ratings.csv',
 'README.txt',
 'tags.csv',
 'links.csv']

In [3]:
# Test of reading the base ratings and movies.
ratings = pd.read_csv("/kaggle/input/datasets/barlettab/movie-lens-32m/ratings.csv")
movies = pd.read_csv("/kaggle/input/datasets/barlettab/movie-lens-32m/movies.csv")

df = ratings.merge(movies, on="movieId")

### • Initial Exploratory Data Analysis (EDA)
#### `ratings` dataframe

In [4]:
# Getting the size of the ratings and movies dataset
ratings.shape, movies.shape

((32000204, 4), (87585, 3))

In [5]:
# Basics informations about ratings dataframe
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [6]:
# Checking ratings dataframe
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [7]:
# Checking if there is null values
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

#### `movies` dataframe

In [8]:
# Basics informations about movies dataframe
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  87585 non-null  int64 
 1   title    87585 non-null  object
 2   genres   87585 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB


In [9]:
# Checking movies dataframe
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [10]:
# Checking if there is null values
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [11]:
# Checking if there are duplicated ids 
ratings['userId'].nunique(), ratings['movieId'].nunique()

(200948, 84432)

## Creating the Baseline
Objective:
- Calculate the average rating.
- Obtain the number of reviews.
  
As a starting point, a popularity-based recommendation system can be built by combining the average rating and the number of votes. This type of approach is widely used as a baseline in real-world recommendation systems before applying more complex models.

In [12]:
# Grouping ratings by movie to calculate performance metrics
# media_rating: the average score given by users
# num_avaliacoes: total count of ratings, representing the movie's popularity
baseline = ratings.groupby("movieId").agg(
    media_rating=("rating", "mean"),
    num_avaliacoes=("rating", "count")
).reset_index()

In [13]:
# Merging base ratings with movies
baseline = baseline.merge(movies, on="movieId")

In [14]:
# Calculating a weighted score for ranking
baseline["score"] = (
    baseline["media_rating"] * baseline["num_avaliacoes"]
)

In [15]:
# Sort the recommendations
baseline_recomendado = baseline.sort_values(
    by="score",
    ascending=False
)

In [16]:
# Displaying the top 5 recommended movies based on the popularity baseline
baseline_recomendado[["title", "media_rating", "num_avaliacoes", "score"]].head(5)

,title,media_rating,num_avaliacoes,score
314,"Shawshank Redemption, The (1994)",4.404614,102929,453362.5
292,Pulp Fiction (1994),4.196969,98409,413019.5
351,Forrest Gump (1994),4.052744,100296,406474.0
2480,"Matrix, The (1999)",4.156437,93808,389907.0
585,"Silence of the Lambs, The (1991)",4.148367,90330,374722.0


In [17]:
# Function to return the top N most popular movies
def recomendar_populares(n=5):
    return baseline_recomendado[[
        "title", "media_rating", "num_avaliacoes"
    ]].head(n)

In [18]:
recomendar_populares(5)

,title,media_rating,num_avaliacoes
314,"Shawshank Redemption, The (1994)",4.404614,102929
292,Pulp Fiction (1994),4.196969,98409
351,Forrest Gump (1994),4.052744,100296
2480,"Matrix, The (1999)",4.156437,93808
585,"Silence of the Lambs, The (1991)",4.148367,90330


It is important to note that this approach, where the score is calculated by multiplying the average rating by the number of reviews, contributes to a popularity bias. In other words, we recommend highly popular movies while potentially overlooking less popular titles that could also be excellent recommendations."

## KNN (K-Nearest Neighbors)

The K-Nearest Neighbors (KNN) model was used to identify movies with similar rating patterns. Each movie is represented as a vector of user interactions, and the similarity between movies is calculated based on the distance between these vectors. Thus, the system recommends movies that have been consumed by users with similar behavior.

- The idea is to answer the following question: "Which movies are watched by the same people?"

In [19]:
# Filtering the dataset to reduce sparsity and improve model quality
# user_counts: identifying how many ratings each user has provided
# movie_counts: identifying how many ratings each movie has received
user_counts = df['userId'].value_counts()
movie_counts = df['movieId'].value_counts()

# Keeping only 'active' users and 'popular' movies (threshold > 500)
# This optimizes memory usage and ensures more reliable similarity calculations
df = df[
    df['userId'].isin(user_counts[user_counts > 500].index) &
    df['movieId'].isin(movie_counts[movie_counts > 500].index)
]

In [20]:
# Memory optimization: Downcasting ratings to float32
df['rating'] = df['rating'].astype('float32')

In [21]:
# Reshaping the dataframe into a User-Item Utility Matrix
user_movie = df.pivot_table(
    index="userId",
    columns="movieId",   
    values="rating"
).fillna(0)

In [22]:
# Initializing the KNN model with Cosine Similarity
model = NearestNeighbors(metric="cosine", algorithm="brute")

# Fitting the model on the transposed matrix (movies as rows)
model.fit(user_movie.T)

NearestNeighbors(algorithm='brute', metric='cosine')

#### Mappers
To optimize processing and ensure the integrity of references, we created mapping dictionaries (mappers) that convert movie IDs into titles and vice versa.

In [23]:
# Creating mapping dictionaries for easy title-to-ID lookup
movie_ids = user_movie.columns

id_to_title = dict(zip(movies['movieId'], movies['title']))
title_to_id = dict(zip(movies['title'], movies['movieId']))

In [24]:
# Function to recommend N movies similar to a given title
def recomendar_filmes(filme, n=5):
    # Converts the title string to its internal numeric ID
    filme_id = title_to_id[filme]

    # Locates the index of the movie in our utility matrix
    idx = list(movie_ids).index(filme_id)

    # Retrieves the movie vector and reshapes it for the KNN model
    # (1, -1) indicates one sample with as many features as there are users
    vetor_filme = user_movie.T.iloc[idx].values.reshape(1, -1)

    # Finds the n_neighbors closest to the input movie
    distances, indices = model.kneighbors(
        vetor_filme,
        n_neighbors=n+1
    )

    recomendados = []
    
    # Maps the neighbor indices back to movie titles
    for i in indices[0]:
        movie_id = movie_ids[i]
        recomendados.append(id_to_title[movie_id])

    # Returns the list excluding the first item (the input movie itself)
    return recomendados[1:]

In [25]:
recomendar_filmes("Toy Story (1995)")

['Toy Story 2 (1999)',
 'Back to the Future (1985)',
 'Forrest Gump (1994)',
 'Jurassic Park (1993)',
 'Matrix, The (1999)']

In [26]:
recomendar_filmes("Twilight (2008)")

['Twilight Saga: New Moon, The (2009)',
 'Twilight Saga: Eclipse, The (2010)',
 'Twilight Saga: Breaking Dawn - Part 1, The (2011)',
 'Twilight Saga: Breaking Dawn - Part 2, The (2012)',
 'Harry Potter and the Half-Blood Prince (2009)']

In [27]:
recomendar_filmes("Shrek (2001)")

['Lord of the Rings: The Fellowship of the Ring, The (2001)',
 'Monsters, Inc. (2001)',
 'Finding Nemo (2003)',
 'Matrix, The (1999)',
 'Lord of the Rings: The Two Towers, The (2002)']

## User-Based Recommendation
- "Which movies has this user not seen yet, but will likely enjoy?"

To answer this question, the system analyzes the user's history to identify which movies they have watched and the ratings they provided. Based on this data, it finds similar movies using the KNN algorithm and then generates personalized recommendations for the user.

In [28]:
# Pre-calculating similarities for all movies at once for performance
movie_ids = user_movie.columns

# Batch processing: Finding the 20 nearest neighbors for every movie in the matrix
# n_neighbors=21 includes the movie itself + 20 similar titles
distances_all, indices_all = model.kneighbors(
    user_movie.T,
    n_neighbors=21
)

# Creating a fast lookup index
movie_index = {movie_id: i for i, movie_id in enumerate(movie_ids)}

In [29]:
# Function to generate personalized recommendations for a specific user
def recomendar_para_usuario(user_id, n=5):
    # Error handling for users not present in the filtered utility matrix
    if user_id not in user_movie.index:
            return ["Usuário não encontrado no modelo reduzido."]

    # Retrieving movies the user has already rated (value > 0)
    filmes_assistidos = user_movie.loc[user_id]
    filmes_assistidos = filmes_assistidos[filmes_assistidos > 0].index

    recomendacoes = {}
    
    # Iterating through history to find neighbors of all watched movies
    for filme_id in filmes_assistidos:

        idx = movie_index[filme_id]

        vizinhos = indices_all[idx]
        distancias = distances_all[idx]

        for i, dist in zip(vizinhos, distancias):

            movie_id = movie_ids[i]

            # Skipping movies the user has already watched
            if movie_id in filmes_assistidos:
                continue

            if movie_id not in recomendacoes:
                recomendacoes[movie_id] = 0

            # Cumulative scoring: more relevant movies appear more frequently as neighbors
            recomendacoes[movie_id] += 1 - dist

    top = sorted(recomendacoes.items(), key=lambda x: x[1], reverse=True)[:n]

    return [id_to_title[movie_id] for movie_id, _ in top]

In [30]:
print(user_movie.index[:5])

Index([10, 28, 35, 65, 70], dtype='int64', name='userId')


In [31]:
recomendar_para_usuario(2, 5)

['Usuário não encontrado no modelo reduzido.']

In [32]:
recomendar_para_usuario(360, 5)

['Mad Max: Fury Road (2015)',
 'Speed (1994)',
 'Being John Malkovich (1999)',
 '300 (2007)',
 'Logan (2017)']

## KNN Recommender Logic
The system utilizes the K-Nearest Neighbors (KNN) algorithm to find similarities between movies based on user behavior.

**1.** Vector Representation: Each movie is represented as a vector of user ratings.

**2.** Distance Calculation: KNN calculates the mathematical distance between these vectors (using metrics like Cosine Similarity).

**3.** Identifying Neighbors: Movies with similar interaction patterns are identified as "neighbors."

**4.** Personalized Mapping: For each user, the system recommends movies that are geographically close in the vector space to the ones they have already watched.


## Model Serialization
At the end of the process, the recommendation system is saved using joblib, ensuring that the entire pipeline can be reused without the need for data reprocessing.

This includes the trained model, auxiliary matrices, and mapping structures.

In [33]:
artefatos = {
    "model": model,
    "user_movie": user_movie,
    "movie_ids": movie_ids,
    "movie_index": movie_index,
    "id_to_title": id_to_title,
    "title_to_id": title_to_id,
    "distances_all": distances_all,
    "indices_all": indices_all
}

In [34]:
joblib.dump(artefatos, "recomendador_movies_knn.sav")

['recomendador_movies_knn.sav']

In [35]:
joblib.dump(artefatos, "/kaggle/working/recomendador_movies_knn.sav")

['/kaggle/working/recomendador_movies_knn.sav']

In [36]:
os.listdir("/kaggle/input")

['datasets']

In [37]:
os.listdir("/kaggle/working")

['.virtual_documents', 'recomendador_movies_knn.sav']